# Día 4 · Unidad 4 — Funciones (III): namespaces y scope

**Objetivos de aprendizaje**
- Explicar qué es un *namespace* y qué tipos existen en Python (`built-in`, global, local, *enclosing*).
- Aplicar la regla LEGB para saber dónde busca Python el valor de una variable.
- Reconocer por qué reasignar una variable "global" dentro de una función crea, en realidad, una variable local distinta.

**Duración estimada:** 45 minutos.

**Requisitos previos:** `01_teoria_funciones_basicas.ipynb` y `02_teoria_args_kwargs.ipynb` de este mismo día.

## 1. Qué es un *namespace*

Un **namespace** es una colección de nombres, cada uno vinculado a un objeto. Puedes pensarlo como un diccionario donde las claves son los nombres de las variables/funciones y los valores son los objetos a los que apuntan.

En Python, en un momento dado, coexisten varios namespaces distintos:

1. **Built-in**: los nombres que Python trae de fábrica (`int`, `list`, `len`, `print`...).
2. **Global**: los nombres definidos en el nivel principal de un módulo o notebook.
3. **Local**: los nombres definidos dentro de una función, mientras se está ejecutando.
4. **Enclosing**: los nombres de una función "que envuelve" a otra función anidada dentro de ella.

### El namespace *built-in*

In [ ]:
nombres_built_in = dir(__builtins__)
print(len(nombres_built_in), "nombres built-in disponibles")
print(nombres_built_in[:15])


> ⚠️ **Error típico**: escribir `dir(__builtin__)`, sin la `s` final. Es un error muy fácil de cometer — existe un módulo interno llamado `__builtin__` en Python 2, pero en Python 3 el namespace accesible se llama `__builtins__`, **con "s" y con dobles guiones bajos a cada lado**. Si lo escribes mal obtienes un `NameError`, porque `__builtin__` (sin "s") no existe como nombre accesible directamente.

In [ ]:
try:
    dir(__builtin__)  # sin la 's' -- no existe como nombre accesible en Python 3
except NameError as e:
    print(f"{type(e).__name__}: {e}")


Entre los nombres de `dir(__builtins__)` reconoces `int`, `list`, `len`, `print`... están disponibles en cualquier punto del programa sin necesidad de `import`.

### El namespace global

Contiene los nombres definidos en el nivel principal del programa (o del notebook). `globals()` lo devuelve como un diccionario:

In [ ]:
tasa_recargo = 0.05
nombre_producto = "auto"

print("tasa_recargo" in globals())
print(globals()["tasa_recargo"])


### El namespace local

Cada vez que se ejecuta una función, Python crea un namespace local nuevo, que sólo existe mientras la función se está ejecutando.

In [ ]:
def calcular_prima_con_recargo(prima_base):
    prima_ajustada = prima_base * (1 + tasa_recargo)
    print("Namespace local dentro de la función:", locals())
    return prima_ajustada


calcular_prima_con_recargo(300.0)


In [ ]:
print("prima_ajustada" in globals())


`prima_ajustada` sólo existió dentro del namespace local de la función — fuera de ella, en el namespace global, ese nombre no existe.

### El namespace *enclosing*

Cuando defines una función dentro de otra función, la función interna tiene acceso al namespace de la función que la contiene (además del suyo propio):

In [ ]:
def calcular_cartera(primas):
    total = sum(primas)

    def aplicar_recargo():
        return total * (1 + tasa_recargo)

    return aplicar_recargo()


calcular_cartera([300.0, 450.0, 275.0])


## 2. La regla LEGB

Cuando Python necesita resolver el valor de un nombre, lo busca en este orden — sus iniciales dan nombre a la regla:

1. **L**ocal — dentro de la función actual.
2. **E**nclosing — dentro de la función que la envuelve, si la hay.
3. **G**lobal — en el nivel principal del módulo/notebook.
4. **B**uilt-in — en los nombres de Python.

In [ ]:
tasa_recargo = 0.05  # global


def calcular_prima_con_recargo_local(prima_base):
    tasa_recargo = 0.20  # local: "tapa" a la global sólo dentro de esta función
    return prima_base * (1 + tasa_recargo)


print(calcular_prima_con_recargo_local(300.0))  # usa la tasa_recargo local (0.20)
print(tasa_recargo)  # la global no ha cambiado


> ⚠️ **Error típico**: asumir que asignar un valor a una variable "global" dentro de una función la modifica. En realidad, si dentro de una función simplemente **asignas** un valor a un nombre que también existe en el global, Python crea una variable **local nueva** con ese nombre — la global queda intacta, aunque tenga el mismo nombre. Esto sorprende a quien viene de otros lenguajes (o de Excel, donde no hay este matiz).

In [ ]:
contador_polizas_procesadas = 0


def procesar_poliza():
    contador_polizas_procesadas = contador_polizas_procesadas + 1  # esto falla
    return contador_polizas_procesadas


try:
    procesar_poliza()
except UnboundLocalError as e:
    print(f"{type(e).__name__}: {e}")


El error confirma la regla: como hay una asignación a `contador_polizas_procesadas` **dentro** de la función, Python la trata como variable local en toda la función — incluida la parte derecha de la asignación, donde intenta *leerla* antes de haberle asignado nada. Si de verdad quieres modificar la variable global desde dentro de una función, hace falta declararlo explícitamente con `global`:

In [ ]:
contador_polizas_procesadas = 0


def procesar_poliza():
    global contador_polizas_procesadas
    contador_polizas_procesadas = contador_polizas_procesadas + 1
    return contador_polizas_procesadas


print(procesar_poliza())
print(procesar_poliza())
print("Valor global final:", contador_polizas_procesadas)


En la práctica, usar `global` casi siempre es señal de que conviene rediseñar la función para que reciba el valor como parámetro y lo devuelva con `return`, en vez de depender de una variable externa — es más fácil de entender, de probar y de reutilizar. Se muestra aquí sobre todo para que reconozcas el patrón si te lo encuentras en código ajeno.

**Para ti:** define una variable global `umbral_alerta = 1000.0`. Escribe una función `superar_umbral(importe)` que compare `importe` con `umbral_alerta` **sin** usar `global` (sólo lectura) y devuelva `True`/`False`. Después, escribe una segunda función `actualizar_umbral(nuevo_valor)` que sí necesite `global` porque *reasigna* `umbral_alerta`.

**Para ti:** ejecuta `def f(): print(x); x = 1` sin haber definido `x` antes en ningún sitio, dentro de un `try/except NameError`. ¿El error que obtienes es `NameError` o `UnboundLocalError`? Relaciónalo con la regla LEGB: ¿por qué Python no busca `x` en el namespace global, si técnicamente el `print(x)` va antes que la asignación local?

## Resumen

- Un namespace es una colección de nombres vinculados a objetos; en Python conviven cuatro: *built-in*, global, local y *enclosing*.
- El namespace correcto en Python 3 es `__builtins__` (con "s"), no `__builtin__` — un error de escritura muy fácil de cometer que produce `NameError`.
- La regla LEGB (Local → Enclosing → Global → Built-in) determina en qué namespace busca Python un nombre, en ese orden.
- Definir una variable con el mismo nombre dentro de una función crea una variable **local** que sólo existe dentro de esa función, y no modifica la global aunque se llame igual.
- Si dentro de una función hay una asignación a un nombre, Python trata ese nombre como local en **toda** la función — incluso antes de la línea de asignación, lo que puede producir `UnboundLocalError` si intentas leerlo antes.
- `global` permite modificar de verdad una variable global desde dentro de una función, pero en la práctica suele ser preferible pasar el valor como parámetro y devolverlo con `return`.

**Siguiente:** `04_teoria_lambda_map_filter.ipynb`, para funciones "de una línea" y su combinación con `map`/`filter`.